# DKT Pipeline — `notebook2.ipynb`

End-to-end Deep Knowledge Tracing pipeline on the student-interaction dataset.

**Task.** For each (user, skill) interaction, predict the evaluation of the *next* attempt as one of `{WRONG, PARTIAL, CORRECT}`.

**Approach.** A per-user sequence model (LSTM) that, at each timestep, emits a 3-way ordinal distribution over the *next* skill the student will attempt. Output is gathered with a one-hot mask of the next skill ID, so we score only the skill the student actually saw next.

**Stages.**
1. Imports & configuration
2. Loading & document universe
3. Cleaning the topic tree
4. Preprocessing transactions
5. Building the feature matrix
6. Splitting and building TensorFlow datasets
7. The DKT model
8. Hyperparameter tuning
9. Training
10. Evaluation
11. Closing notes


## 1. Imports & Configuration

All hyperparameters and dataset-level toggles are concentrated below so every knob the pipeline exposes is visible in one place before any code runs.

- `SUBJECTS` lists the two subjects trained sequentially in a single notebook run (`'math'` and `'german'`).
- `RANDOM_STATE` controls the user-level split.
- `params` carries the base hyperparameters consumed by every model: batch size, LSTM capacity, dropout, optimizer, and epoch budget. `recurrent_units` and `dropout_rate` are starting-point values here — §8 overwrites them per subject with the tuning winner. Weight checkpoint paths are derived per subject in §6 (`weights/{subject}_bestmodel.weights.h5`).

In [1]:
from src.data import *
from src.debug import print_dataset_info
from src.features import *
from src.models import *
from src.models import _flatten

import collections
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn import feature_extraction, model_selection
from sklearn.metrics import mean_squared_error, roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import MinMaxScaler


In [2]:
# Pipeline configuration
from src.config import MASK_VALUE

# Topic-tree surgery (see §3 for rationale).
TOPICS_TO_REPARENT = [2026, 2027, 2028]
REPARENT_UNDER     = 1
TOPICS_TO_DROP     = [2029, 3425]

# Subjects to train.
SUBJECTS     = ['math', 'german']
RANDOM_STATE = 0

# Set to True to skip retraining and load existing checkpoint weights instead.
# Allows running §10 (evaluation) without repeating the full training run.
SKIP_TRAINING = True

# Base training / model hyperparameters (per-subject best values applied in §8).
# recurrent_units=256 and dropout_rate=0.0: tuning showed that 1024 units lifted
# validation AUC only marginally over 256 while training time grew ~4×; the
# performance gain did not justify the extra compute, so 256 is used throughout.
params = {
    'batch_size'     : 32,
    'mask_value'     : MASK_VALUE,
    'recurrent_units': 256,
    'dropout_rate'   : 0.0,
    'optimizer'      : 'adam',
    'epochs'         : 20,
    'verbose'        : 1,
}

## 2. Loading & Document Universe

`load_all_data()` returns a `dict` of the five raw tables (`documents`, `topic_trees`, `topics_translated`, `transactions`, `users`). The `documents` table has multiple versions per `document_id`; `get_latest_documents()` keeps the latest version per document and parses the JSON `content` column to extract `estimatedDuration` and `estimatedDifficulty`. `summarize_documents()` reports the share of documents that carry both metadata fields — the rest will be silently dropped at the difficulty join in §4.


In [3]:
dfs = load_all_data()


In [4]:
documents = get_latest_documents(documents=dfs['documents'])

In [5]:
summarize_documents(documents)


Total distinct documents: 5746
Documents with estimatedDuration and estimatedDifficulty: 4735 (82.4%)
Filtered out: 1011 (17.6%)
Distinct topics: 370


['fTp7AX3QQfx8vvTLNKGPRv',
 'fqjEP599AZ29AtN0XNgWxT',
 'etfOjQpk4os8hF9z0rMfwt',
 'aVwP8ywckxMbAH-8w2wfMe',
 'cEY-K2TJ4HGaNRqp5bjhF5',
 'bKyFkknHAW7aeH4d0qhpEn',
 '1fsDFKDLk8KbZ2xBBzNx--',
 '8JmheE6M4208C3iLPQHRnP',
 '1eUS4GFW4.g8vLxi-sRo1z',
 'bNPQokamA9sbXPqYi8fSzh',
 'awGP3BNZk5UbMHbTW4R5x3',
 '59scgU0wkw-8qdY1ZP59DI',
 'Af0RVdAEibQAr.M9AVTj',
 'ffTij6YYA-zbcx9ZP8QWLy',
 '4HNL62l.47Ta0sHqrOzKoF',
 'aCF0hfb1kQEayDOqwEp0s1',
 '78debrP047o8YoDB0nymp2',
 'dCZ4cWixQrM9ovYwc-TP.z',
 'Bx98geNrvUYWSjsh8g',
 '6f7aujnwQTxb56qwxs1sFX',
 'bVubBjOtA6x9x3pOLGQL95',
 'ctu8cT8ZQVebZn4G3nux8s',
 'dNMQxmWW4pe9auQHzX6DW7',
 '3D4ovRZd4e69keiP-DEyXl',
 '9XwCwbGcQaObYPjBFfQeab',
 'fkcsHMUrkcvadzh3aQdifI',
 '6shnOMCmAMSasjMJay1KdE',
 '448ge.u7Qzyaj4XuEBTIaH',
 '5oWgAX-Fk82bNdaRxoXBgh',
 '6SUe7t6SkV58p70qoJwl3K',
 '9VnHcCno4dF93oGTP1nYyc',
 '483vpC1jQbk8WYeIpt6T-d',
 '56ljt80hkbH8A4vzX.Je5q',
 '3XEDx-kl4Egb92HW9bD0mC',
 '8TxLI5oe4c29OZzvd4o85a',
 'Li7CRHY4gP98s.i4z1-by',
 '5WUsFXhOQwJ9e8Gm81NMto',
 '7CvyCZ

## 3. Cleaning the Topic Tree

The raw topic tree contains structural noise that hurts the modeling task:
- Several top-level topics that should logically sit under the German root.
- Branches that no document references (dead leaves and dead intermediate nodes).
- A pair of subtrees that are explicitly stub/placeholder content.

We perform three surgical operations in order, each justified below.


### 3.1 Inspect the raw tree

`build_topic_lookups()` returns four dicts (`id_to_name`, `id_to_math`, `child_to_parent`, `parent_to_children`). `add_topic_depth()` copies `topics_translated` and adds a `depth` column by walking parents. The cell below prints the depth histogram and the count of doc-bearing topics per depth; the next cell isolates the depth-0 topics that *actually* host documents — those are the candidates for surgery in §3.2 and §3.4.

In [6]:
lookups = build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])
topics  = add_topic_depth(dfs['topics_translated'], lookups['child_to_parent'])
print(f"\n{len(lookups['child_to_parent'])} child→parent edges")


Depth distribution across all topics:
       n_topics
depth          
0            25
1            72
2           179
3           294
4           110
5            20

675 child→parent edges


In [7]:
# How many topics at each depth actually host documents?
# This is what motivates §3.3: most non-leaf topics carry no documents directly.
display(topics_with_docs_per_depth(documents, topics))


,n_topics_with_docs
depth,
0,7
1,20
2,98
3,187
4,38
5,20


In [8]:
# Depth-0 topics that actually carry documents — candidates for the root or for surgery.
active = documents['topic_id'].unique()
display(topics[(topics['depth'] == 0) & (topics['id'].isin(active))])


,id,german_name,german_description,name,description,math,depth
0,1,Deutsch,Sprache als System,German,Language as a system,0,0
3,109,Mathematik,Rechnen und so...,Mathematics,Calculating and such...,1,0
116,2026,Ober-/Unterbegriff,NaN,Top/sub term,NaN,0,0
117,2027,Synonyme/Antonyme,NaN,Synonyms/antonyms,NaN,0,0
118,2028,Mehrdeutigkeit,NaN,Ambiguity,NaN,0,0
119,2029,Verschiedenes,NaN,Miscellaneous,NaN,0,0
477,3425,Zu löschen,"ungültig, wird gelöscht.",To delete,"invalid, will be deleted.",1,0


### 3.2 Reparent `[2026, 2027, 2028]` under root `1`

The depth-0 listing in §3.1 shows three topics — `Ober-/Unterbegriff` (top/sub term), `Synonyme/Antonyme`, and `Mehrdeutigkeit` (ambiguity) — sitting at the root of the tree. They are German linguistic categories that conceptually belong under the German root (`id=1`, "Sprache als System") but were authored at the top level. Leaving them detached would make the depth feature lie about how specific each topic is, and would split the German subject across multiple disjoint roots when §4.1 partitions by `topics.math`.

`reparent_topics()` ([src/data.py:146](src/data.py#L146)) detaches the listed children and re-inserts them with parent `id=1`, allocating fresh `topic_id`s past the current max so primary keys stay unique.

In [9]:
dfs['topic_trees'] = reparent_topics(
    dfs['topic_trees'], TOPICS_TO_REPARENT, new_parent_id=REPARENT_UNDER,
)
display(dfs['topic_trees'][dfs['topic_trees']['child_id'].isin(TOPICS_TO_REPARENT)])


,topic_id,parent_id,child_id,sibling_rank,displayed_on_dashboard
678,6108,1.0,2026,0,0
679,6109,1.0,2027,0,0
680,6110,1.0,2028,0,0


### 3.3 Prune topics with no documents

`_non_empty_topic_ids` ([src/data.py:167](src/data.py#L167)) starts from the set of topics that *directly* host at least one document and walks parent pointers to collect every ancestor. A topic survives iff it has a descendant document; this drops dead leaves *and* dead intermediate nodes in one pass. We apply the same filter to both `topic_trees` and `topics_translated` so the two stay in sync.


In [10]:
before_tt = len(dfs['topic_trees'])
before_tr = len(dfs['topics_translated'])
dfs['topic_trees']        = prune_empty_topics(dfs['topic_trees'], documents)
dfs['topics_translated']  = prune_empty_topics_translated(
    dfs['topics_translated'], dfs['topic_trees'], documents,
)
print(f"topic_trees rows: {before_tt} -> {len(dfs['topic_trees'])}")
print(f"topics_translated rows: {before_tr} -> {len(dfs['topics_translated'])}")


topic_trees rows: 681 -> 378
topics_translated rows: 700 -> 379


### 3.4 Drop the `[2029, 3425]` subtrees

Two depth-0 topics survive the empty-prune of §3.3 because they still carry a handful of stale documents: `2029 — Verschiedenes` ("Miscellaneous") and `3425 — Zu löschen` (whose own description literally reads *"ungültig, wird gelöscht"* — "invalid, will be deleted"). The author flagged them for removal but the cleanup never happened. We drop both subtrees outright so the model isn't asked to learn mastery curves over content the platform itself considers junk.

`drop_topic_subtrees()` ([src/data.py:213](src/data.py#L213)) does a downward BFS from each root and removes every collected ID from both tables.

In [11]:
before_tt = len(dfs['topic_trees'])
before_tr = len(dfs['topics_translated'])
dfs['topic_trees'], dfs['topics_translated'] = drop_topic_subtrees(
    dfs['topic_trees'], dfs['topics_translated'], TOPICS_TO_DROP,
)
print(f"topic_trees rows: {before_tt} -> {len(dfs['topic_trees'])}")
print(f"topics_translated rows: {before_tr} -> {len(dfs['topics_translated'])}")


topic_trees rows: 378 -> 377
topics_translated rows: 379 -> 377


In [12]:
# Recompute lookups + depth table on the cleaned trees — `topics` is consumed
# by §4 (split_by_subject) and §5 onward.
lookups = build_topic_lookups(dfs['topics_translated'], dfs['topic_trees'])
topics  = add_topic_depth(dfs['topics_translated'], lookups['child_to_parent'])


Depth distribution across all topics:
       n_topics
depth          
0             2
1            25
2           103
3           188
4            39
5            20


## 4. Preprocessing Transactions

`summarize_transactions()` ([src/data.py:240](src/data.py#L240)) does two things at once:
1. Keeps only rows with a non-null `evaluation` (drops in-progress / abandoned attempts).
2. Joins each transaction's `estimatedDifficulty` from `documents` via `document_id`.

It also reports two leakage sources — rows whose `topic_id` is no longer in the cleaned topics table, and rows whose document has no `estimatedDifficulty`. **No rows are dropped here**: filtering is deferred to feature construction, where missing values are coalesced to a `-2` sentinel that becomes part of the skill key.


In [13]:
evaluated = summarize_transactions(dfs['transactions'], topics, documents)
display(evaluated.head())


Total transactions: 2134759
Evaluated transactions: 1401007 (65.6%)
Evaluated with unknown topic_id: 348782 (24.9%)
Evaluated with no estimatedDifficulty: 34228 (2.4%)


,transaction_id,transaction_token,user_id,document_id,document_version,evaluation,input,start_time,commit_time,user_agent,...,session_id,topic_id,session_closed,session_type,session_accepted,challenge,challenge_id,challenge_order,challenge_name,estimatedDifficulty
0,688413,88fdcaad-f73b-46a2-b561-d262f2441442,393211,awd0i1DlVtg6kuMZSkpmHa,75002,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 07:58:27.312000000,2021-05-21 08:03:43.020000000,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,2.0,G3h – Training Rhetorik,3.0
1,688414,a75eb7b4-b2c2-47d4-9200-27980c175037,393211,arhWF3BT53V9W8cGOaZVPX,75012,PARTIAL,"{""type"": ""MULTI_COLOR_HIGHLIGHT"", ""highlighted...",2021-05-21 08:04:05.067000000,2021-05-21 08:07:21.288999936,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,3.0,G3h – Training Rhetorik,4.0
2,688415,61eb829d-bdda-4107-86af-ad9a14a7bdc9,393211,9wk5dtV2mF59odW0wCEYYc,75003,PARTIAL,"{""type"": ""CLOZE_TEXT"", ""clozeInputs"": [""Person...",2021-05-21 08:07:37.048000000,2021-05-21 08:13:30.953999872,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,4.0,G3h – Training Rhetorik,3.0
3,688416,30ff0d8a-865d-460b-9177-b698a52b0d5c,393211,afilxZ8LycP5LReULeKngW,75009,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Ich gehe i...",2021-05-21 08:13:38.943000000,2021-05-21 08:22:13.975000064,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,5.0,G3h – Training Rhetorik,3.0
4,688417,0adedf3b-ba35-4497-8c6b-b5c2f6fcbbf3,393211,76m6v05NCeX8x2Wr5tKRE3,75007,CORRECT,"{""type"": ""DND_PAIRS"", ""input"": [""<p>Kleiner Ma...",2021-05-21 08:22:19.391000000,2021-05-21 08:22:55.366000128,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_6...,...,NaN,NaN,NaN,NaN,NaN,True,1083.0,6.0,G3h – Training Rhetorik,2.0


### 4.1 Split by subject

`split_by_subject()` partitions on `topics.math ∈ {0, 1}`. We pick one subject for the rest of the pipeline; flipping `SUBJECT` in §1 reruns everything against German.


In [14]:
math_df, german_df = split_by_subject(evaluated, topics)
print(f"math: {len(math_df)} rows | german: {len(german_df)} rows")
subject_dfs = {'math': math_df, 'german': german_df}

math: 293465 rows | german: 758760 rows


## 5. Building the Feature Matrix

`build_feature_matrix()` ([src/features.py:97](src/features.py#L97)) produces the modeling table `X` with columns `[user_id, skill_name, correct, start_time, can_partial, time_since_first_attempt, skill_attempts, total_attempts]`.

We run it independently for each subject and store the results in the `X` dict, keyed by subject name. Non-obvious choices:
- **Composite skill key.** `skill_name = topic_id + '_' + estimatedDifficulty` — same topic at a different difficulty is treated as a *different* skill, so the DKT model can learn distinct mastery curves per difficulty band.
- **Sentinel `-2` for missing values.** Both `topic_id` and `estimatedDifficulty` get `fillna(-2)` *before* the skill key is built, so missing-metadata rows form their own pseudo-skills instead of being dropped.
- **`can_partial` flag.** A per-document boolean for whether the question structurally admits a `PARTIAL` evaluation. `_partial_rule()` ([src/features.py:18](src/features.py#L18)) inspects the document `type` and parsed `content`. Documents whose rule says `False` but for which a `PARTIAL` was *observed* are flipped to `True`.
- **`time_since_first_attempt`.** Seconds elapsed between the transaction's `start_time` and the user's earliest answered question.
- **Causal cumulative counters.** `skill_attempts` and `total_attempts` use `cumcount()` per group — no leakage of the current row.

The label `correct` is mapped from `evaluation` strings via `EVAL_CODES = {WRONG: 0, PARTIAL: 1, CORRECT: 2}` ([src/features.py:8](src/features.py#L8)).

In [15]:
X = {}
for subject in SUBJECTS:
    X[subject] = build_feature_matrix(subject_dfs[subject], documents)
    print(f"\n--- {subject} ---")
    display(X[subject].head())


--- math ---


,user_id,skill_name,correct,start_time,can_partial,time_since_first_attempt,skill_attempts,total_attempts
0,390142,1046_2,2,2021-05-21 10:16:09.924000000,False,0.000,0,0
1,390137,1059_1,1,2021-05-21 10:16:58.803000000,True,0.000,0,0
2,390140,987_1,0,2021-05-21 10:20:53.223000000,False,0.000,0,0
3,390140,987_2,0,2021-05-21 10:23:24.005000000,False,150.782,0,1
4,390140,987_1,0,2021-05-21 10:24:10.729000000,False,197.506,1,2



--- german ---


,user_id,skill_name,correct,start_time,can_partial,time_since_first_attempt,skill_attempts,total_attempts
0,393224,3163_3,1,2021-05-21 11:16:29.867000000,True,0.0,0,0
1,388363,3163_1,0,2021-05-21 11:16:46.783000000,True,0.0,0,0
2,393232,3163_3,1,2021-05-21 11:16:54.135000000,True,0.0,0,0
3,393231,3163_3,2,2021-05-21 11:17:01.595000000,True,0.0,0,0
4,393230,3163_3,2,2021-05-21 11:17:19.189000000,True,0.0,0,0


In [16]:
for subject in SUBJECTS:
    print(f"{subject}: {X[subject]['user_id'].nunique()} unique students, "
          f"{X[subject]['skill_name'].nunique()} unique skills")

math: 9110 unique students, 157 unique skills
german: 13096 unique students, 140 unique skills


## 6. Splitting and Building TensorFlow Datasets

Three subtleties drive this section:

**Group-aware split.** We split by *user*, not by interaction, using `GroupShuffleSplit` ([src/models.py:10](src/models.py#L10)) with `random_state=RANDOM_STATE`. We apply the same split twice (80/20 outer, then 80/20 inside train) for an effective **64/16/20 train/val/test** breakdown. This is done independently for both subjects.

**Sequence construction.** `prepare_seq()` ([src/features.py:145](src/features.py#L145)) factorizes `skill_name` with `sort=True` (deterministic), then for each user produces a 5-tuple aligned to the *next* step: past `skill_with_answer`, next skill, next correct label, next `can_partial` flag, and `time_since_first_attempt` in days.

**Per-subject params.** Each subject gets its own copy of `params` with a dedicated checkpoint path (`weights/{subject}_bestmodel.weights.h5`). The `subject_params` dict is the single source of truth for all downstream code.

**Padding & repetition.** `prepare_data()` ([src/models.py:26](src/models.py#L26)) one-hot encodes the past-features tensor, appends the two auxiliary scalars, pads each batch to its longest sequence, and `repeat()`s the dataset.

In [17]:
# Per-subject 64/16/20 train/val/test split (by user, not interaction).
splits = {}
for subject in SUBJECTS:
    train_index, test_index = next(create_iterator(X[subject]))
    X_train = X[subject].iloc[train_index]
    X_test  = X[subject].iloc[test_index]
    train_val_index, val_index = next(create_iterator(X_train))
    X_train_val = X_train.iloc[train_val_index]
    X_val       = X_train.iloc[val_index]
    splits[subject] = dict(X_train_val=X_train_val, X_val=X_val, X_test=X_test)
    print(f"{subject}: {len(X_train_val)} train | {len(X_val)} val | {len(X_test)} test interactions")

math: 176803 train | 48853 val | 67809 test interactions
german: 484232 train | 127559 val | 146969 test interactions


In [18]:
# Build TF datasets and per-subject params for train / val / test.
datasets       = {}
subject_params = {}

for subject in SUBJECTS:
    sp = splits[subject]
    seq, features_depth, skill_depth = prepare_seq(X[subject])

    seq_train = seq[sp['X_train_val'].user_id.unique()]
    seq_val   = seq[sp['X_val'].user_id.unique()]
    seq_test  = seq[sp['X_test'].user_id.unique()]

    p = {
        **params,
        'best_model_weights': f'weights/{subject}_bestmodel.weights.h5',
    }
    tf_train, length      = prepare_data(seq_train, p, features_depth, skill_depth)
    tf_val,   val_length  = prepare_data(seq_val,   p, features_depth, skill_depth)
    tf_test,  test_length = prepare_data(seq_test,  p, features_depth, skill_depth)

    p['train_size'] = int(length      // p['batch_size'])
    p['val_size']   = int(val_length  // p['batch_size'])
    p['test_size']  = int(test_length // p['batch_size'])

    datasets[subject] = dict(
        train=tf_train, val=tf_val, test=tf_test,
        features_depth=features_depth, skill_depth=skill_depth,
    )
    subject_params[subject] = p
    print(f"{subject}: {p['train_size']} train | {p['val_size']} val | {p['test_size']} test batches")

2026-04-29 23:09:44.028509: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2026-04-29 23:09:44.028682: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-04-29 23:09:44.028710: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-04-29 23:09:44.028784: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-04-29 23:09:44.028812: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


math: 182 train | 45 val | 56 test batches
german: 261 train | 65 val | 81 test batches


## 7. The DKT Model

`create_model_lstm()` ([src/models.py:102](src/models.py#L102)) wires up:

```
features (B, T, features_depth + 2) ─► LSTM(recurrent_units, return_sequences=True, dropout)
                                    ─► TimeDistributed(Dense(n_skills × N_EVAL_STATES))
                                    ─► Reshape to (B, T, n_skills, N_EVAL_STATES)
                                    ─► GatherSkill: einsum with one-hot(next_skill)
                                    ─► output (B, T, N_EVAL_STATES)
```

The `+ 2` on the input width is the auxiliary `can_partial` and `time_since_first_days` scalars concatenated to the one-hot `skill_with_answer`.

The `GatherSkill` layer ([src/models.py:88](src/models.py#L88)) keeps the loss focused: at each timestep we emit a 3-vector for *every* skill, but the loss is computed only on the row corresponding to the skill the student actually attempted next.

**Metrics.** `AUC`: macro one-vs-rest across the 3 ordinal classes. `RMSE`: ordinal RMSE between true class and softmax-expected value. `accuracy`: sparse categorical accuracy on the argmax.

The cell below prints the architecture using the **math** subject dimensions as a representative example. The german model has the same structure but different `features_depth` and `skill_depth` values.

> This preview uses the base `params` values; §9 rebuilds each model from scratch after §8 has updated `subject_params` with the per-subject tuning winner.

In [19]:
# Architecture preview using math dimensions as a representative example.
fd_math = datasets['math']['features_depth']
sd_math = datasets['math']['skill_depth']
dkt_preview = create_model_lstm(fd_math, sd_math, subject_params['math'])
dkt_preview.summary()

Model: "DKT"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 features (InputLayer)       [(None, None, 473)]          0         []                            
                                                                                                  
 lstm (LSTM)                 (None, None, 256)            747520    ['features[0][0]']            
                                                                                                  
 time_distributed (TimeDist  (None, None, 468)            120276    ['lstm[0][0]']                
 ributed)                                                                                         
                                                                                                  
 reshape (Reshape)           (None, None, 156, 3)         0         ['time_distributed[0][0]']  

## 8. Hyperparameter Tuning

Before committing to a 20-epoch run we sweep `recurrent_units` (model capacity) and `dropout_rate` (regularization) for **each subject separately** and pick the configuration with the best validation AUC. Each trial is trained for `TUNING_EPOCHS` epochs; the assumption is that the *ranking* of configs after a few epochs is a decent proxy for their final ranking.

Trials are checkpointed to `weights/{subject}_tuning_results.json` after every config so a crash mid-grid doesn't lose prior work; on restart, configs already in the file are skipped. After each trial we drop the model and clear the Keras session so TF releases the graph and weights.

The winning config per subject is written back into `subject_params[subject]`, so §9's training runs automatically use them.

In [20]:
from sklearn.model_selection import ParameterGrid

TUNING_EPOCHS = 5

search_grid = {
    'recurrent_units': [16, 64, 256],
    'dropout_rate':    [0.0, 0.1, 0.3],
}

configs = list(ParameterGrid(search_grid))

def _cfg_key(d):
    return tuple(sorted((k, d[k]) for k in search_grid))

print(f"{len(configs)} configurations × {TUNING_EPOCHS} epochs each")

9 configurations × 5 epochs each


In [21]:
import gc
import json
import os
import time

all_results = {}

for subject in SUBJECTS:
    TUNING_RESULTS_PATH = f'weights/{subject}_tuning_results.json'
    os.makedirs(os.path.dirname(TUNING_RESULTS_PATH), exist_ok=True)

    if os.path.exists(TUNING_RESULTS_PATH):
        with open(TUNING_RESULTS_PATH) as f:
            results = json.load(f)
        print(f"\n[{subject}] Resuming from {TUNING_RESULTS_PATH} ({len(results)} trial(s) already done)")
    else:
        results = []

    done_keys = {_cfg_key({k: r[k] for k in search_grid}) for r in results}
    p         = subject_params[subject]
    fd        = datasets[subject]['features_depth']
    sd        = datasets[subject]['skill_depth']
    tf_tr     = datasets[subject]['train']
    tf_va     = datasets[subject]['val']
    n_trials  = len(configs)

    for i, cfg in enumerate(configs, start=1):
        if _cfg_key(cfg) in done_keys:
            print(f"  [{subject}] Trial {i}/{n_trials} :: {cfg} -- already done, skipping")
            continue
        trial_params = {**p, **cfg, 'epochs': TUNING_EPOCHS, 'verbose': 2}
        print(f"\n  [{subject}] Trial {i}/{n_trials} :: {cfg}")
        t0      = time.time()
        model   = create_model_lstm(fd, sd, trial_params)
        history = model.fit(
            tf_tr,
            epochs=trial_params['epochs'],
            steps_per_epoch=trial_params['train_size'],
            validation_data=tf_va,
            validation_steps=trial_params['val_size'],
            verbose=2,
        )
        val_auc  = max(history.history['val_auc'])
        val_loss = min(history.history['val_loss'])
        elapsed  = time.time() - t0
        results.append({**cfg, 'val_auc': float(val_auc), 'val_loss': float(val_loss)})
        print(f"  --> Trial {i}/{n_trials} done in {elapsed:.1f}s  "
              f"best val_auc={val_auc:.4f}  best val_loss={val_loss:.4f}")

        with open(TUNING_RESULTS_PATH, 'w') as f:
            json.dump(results, f, indent=2)

        del model, history
        tf.keras.backend.clear_session()
        gc.collect()

    results_df = (
        pd.DataFrame(results)
          .sort_values('val_auc', ascending=False)
          .reset_index(drop=True)
    )
    print(f"\n=== [{subject}] Tuning summary (sorted by val_auc) ===")
    display(results_df)
    all_results[subject] = results_df


[math] Resuming from weights/math_tuning_results.json (10 trial(s) already done)
  [math] Trial 1/9 :: {'dropout_rate': 0.0, 'recurrent_units': 16} -- already done, skipping
  [math] Trial 2/9 :: {'dropout_rate': 0.0, 'recurrent_units': 64} -- already done, skipping
  [math] Trial 3/9 :: {'dropout_rate': 0.0, 'recurrent_units': 256} -- already done, skipping
  [math] Trial 4/9 :: {'dropout_rate': 0.1, 'recurrent_units': 16} -- already done, skipping
  [math] Trial 5/9 :: {'dropout_rate': 0.1, 'recurrent_units': 64} -- already done, skipping
  [math] Trial 6/9 :: {'dropout_rate': 0.1, 'recurrent_units': 256} -- already done, skipping
  [math] Trial 7/9 :: {'dropout_rate': 0.3, 'recurrent_units': 16} -- already done, skipping
  [math] Trial 8/9 :: {'dropout_rate': 0.3, 'recurrent_units': 64} -- already done, skipping
  [math] Trial 9/9 :: {'dropout_rate': 0.3, 'recurrent_units': 256} -- already done, skipping

=== [math] Tuning summary (sorted by val_auc) ===


,dropout_rate,recurrent_units,val_auc,val_loss
0,0.0,1024,0.709861,0.150606
1,0.0,256,0.709350,0.150594
2,0.1,256,0.703161,0.151651
3,0.1,64,0.701023,0.152275
4,0.3,256,0.699619,0.151533
5,0.3,64,0.695608,0.152787
6,0.0,64,0.690242,0.153619
7,0.3,16,0.683525,0.154671
8,0.1,16,0.671266,0.156041
9,0.0,16,0.669673,0.156134



[german] Resuming from weights/german_tuning_results.json (10 trial(s) already done)
  [german] Trial 1/9 :: {'dropout_rate': 0.0, 'recurrent_units': 16} -- already done, skipping
  [german] Trial 2/9 :: {'dropout_rate': 0.0, 'recurrent_units': 64} -- already done, skipping
  [german] Trial 3/9 :: {'dropout_rate': 0.0, 'recurrent_units': 256} -- already done, skipping
  [german] Trial 4/9 :: {'dropout_rate': 0.1, 'recurrent_units': 16} -- already done, skipping
  [german] Trial 5/9 :: {'dropout_rate': 0.1, 'recurrent_units': 64} -- already done, skipping
  [german] Trial 6/9 :: {'dropout_rate': 0.1, 'recurrent_units': 256} -- already done, skipping
  [german] Trial 7/9 :: {'dropout_rate': 0.3, 'recurrent_units': 16} -- already done, skipping
  [german] Trial 8/9 :: {'dropout_rate': 0.3, 'recurrent_units': 64} -- already done, skipping
  [german] Trial 9/9 :: {'dropout_rate': 0.3, 'recurrent_units': 256} -- already done, skipping

=== [german] Tuning summary (sorted by val_auc) ===


,dropout_rate,recurrent_units,val_auc,val_loss
0,0.0,1024,0.733852,0.138009
1,0.0,256,0.731827,0.137798
2,0.0,64,0.728053,0.140015
3,0.1,256,0.717554,0.141036
4,0.1,64,0.707075,0.144299
5,0.3,256,0.702124,0.144483
6,0.3,64,0.679133,0.148375
7,0.3,16,0.677342,0.149498
8,0.1,16,0.675641,0.149870
9,0.0,16,0.674132,0.150121


In [22]:
# Promote the winning config per subject into subject_params.
for subject in SUBJECTS:
    best = all_results[subject].iloc[0]
    subject_params[subject]['recurrent_units'] = int(best['recurrent_units'])
    subject_params[subject]['dropout_rate']    = float(best['dropout_rate'])
    print(f"[{subject}] Best config: recurrent_units={subject_params[subject]['recurrent_units']}, "
          f"dropout_rate={subject_params[subject]['dropout_rate']}")
    print(f"[{subject}] Best val_auc over {TUNING_EPOCHS}-epoch search: {best['val_auc']:.4f}")

[math] Best config: recurrent_units=1024, dropout_rate=0.0
[math] Best val_auc over 5-epoch search: 0.7099
[german] Best config: recurrent_units=1024, dropout_rate=0.0
[german] Best val_auc over 5-epoch search: 0.7339


In [23]:
# Although tuning identified 1024 recurrent units as the top config for both
# subjects, the AUC gain over 256 units was negligible (~0.002) while training
# time grew roughly 4×. We therefore override both subjects to recurrent_units=256
# and dropout_rate=0.0 as the best compute-performance trade-off.
for subject in SUBJECTS:
    subject_params[subject]['recurrent_units'] = 256
    subject_params[subject]['dropout_rate']    = 0.0
    print(f"[{subject}] Overridden to recurrent_units=256, dropout_rate=0.0")

[math] Overridden to recurrent_units=256, dropout_rate=0.0
[german] Overridden to recurrent_units=256, dropout_rate=0.0


## 9. Training

`train_dkt()` ([src/models.py:53](src/models.py#L53)) wires up a `ModelCheckpoint` that saves only the best-validation weights and runs `model.fit` against the repeating train/val streams. We train both subjects sequentially, each with the winning hyperparameters from §8. Checkpoints land in `weights/{subject}_bestmodel.weights.h5`.

Set `SKIP_TRAINING = True` in §1 to bypass retraining entirely: the cell builds each model architecture and immediately loads the saved checkpoint, so §10 (evaluation) can be run without repeating the full training run.

**Chosen configuration: `recurrent_units=256`, `dropout_rate=0.0`.** Tuning showed that scaling to 1024 units yielded only a marginal improvement in validation AUC over 256, while increasing training time roughly 4×. Given that the gain in predictive performance was not commensurate with the additional computational cost, 256 recurrent units with no dropout was selected as the best trade-off between model capacity and efficiency.

In [24]:
models    = {}
histories = {}

for subject in SUBJECTS:
    p   = subject_params[subject]
    fd  = datasets[subject]['features_depth']
    sd  = datasets[subject]['skill_depth']
    dkt = create_model_lstm(fd, sd, p)

    if SKIP_TRAINING:
        dkt.load_weights(p['best_model_weights'])
        print(f"[{subject}] Loaded weights from {p['best_model_weights']} (SKIP_TRAINING=True)")
        histories[subject] = None
    else:
        print(f"\n{'='*60}")
        print(f"  Training {subject} model")
        print(f"{'='*60}")
        history = train_dkt(dkt, datasets[subject]['train'], datasets[subject]['val'], p)
        histories[subject] = history

    models[subject] = dkt

[math] Loaded weights from weights/math_bestmodel.weights.h5 (SKIP_TRAINING=True)
[german] Loaded weights from weights/german_bestmodel.weights.h5 (SKIP_TRAINING=True)


## 10. Evaluation

We reload each subject's best-validation weights and score on the respective held-out test users. `model.evaluate` returns headline metrics; §10.1 builds a per-class confusion matrix for each subject to inspect the shape of the model's errors.

In [25]:
for subject in SUBJECTS:
    print(f"\n{'='*60}")
    print(f"  Evaluating {subject} model")
    print(f"{'='*60}")
    p = subject_params[subject]
    models[subject].load_weights(p['best_model_weights'])
    models[subject].evaluate(
        datasets[subject]['test'],
        steps=p['test_size'],
        verbose=p['verbose'],
        return_dict=True,
    )


  Evaluating math model


2026-04-29 23:09:46.365158: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


56/56 [==============================] - 17s 298ms/step - loss: 0.1307 - auc: 0.7406 - root_mean_squared_error: 0.8153 - accuracy: 0.6151

  Evaluating german model
81/81 [==============================] - 13s 155ms/step - loss: 0.1447 - auc: 0.7531 - root_mean_squared_error: 0.5678 - accuracy: 0.6255


### 10.1 Confusion matrices (row-normalized)

One matrix per subject. Rows are true classes; each row sums to 1. The diagonal is per-class recall; off-diagonal cells reveal which substitutions the model is making. For an ordinal task, off-by-one errors (e.g. predicting `PARTIAL` for `CORRECT`) are far less concerning than flipping `WRONG ↔ CORRECT`.

In [26]:
names = ['WRONG', 'PARTIAL', 'CORRECT']

for subject in SUBJECTS:
    print(f"\n{'='*60}")
    print(f"  Confusion matrix — {subject}")
    print(f"{'='*60}")
    p = subject_params[subject]
    true_all, pred_all = [], []
    for (inputs, lbl, w) in datasets[subject]['test'].take(p['test_size']):
        keep = w.numpy() > 0
        pred = tf.argmax(models[subject](inputs), axis=-1).numpy()
        true_all.extend(lbl.numpy()[keep].tolist())
        pred_all.extend(pred[keep].tolist())

    true_all = np.array(true_all)
    pred_all = np.array(pred_all)

    print("true distribution:", collections.Counter(true_all))
    print("pred distribution:", collections.Counter(pred_all))

    cm = pd.crosstab(
        pd.Series(true_all, name='true'),
        pd.Series(pred_all, name='pred'),
        normalize='index',
    ).round(3)
    cm.index   = [names[i] for i in cm.index]
    cm.columns = [names[i] for i in cm.columns]
    print(cm)


  Confusion matrix — math
true distribution: Counter({2: 38798, 0: 15911, 1: 10504})
pred distribution: Counter({2: 52413, 0: 7349, 1: 5451})
         WRONG  PARTIAL  CORRECT
WRONG    0.233    0.040    0.727
PARTIAL  0.041    0.268    0.691
CORRECT  0.083    0.052    0.866

  Confusion matrix — german
true distribution: Counter({1: 73205, 2: 60577, 0: 10203})
pred distribution: Counter({1: 87464, 2: 49492, 0: 7029})
         WRONG  PARTIAL  CORRECT
WRONG    0.215    0.434    0.351
PARTIAL  0.033    0.770    0.197
CORRECT  0.040    0.440    0.520


## 11. Closing Notes

- **Reproducibility.** Splits are deterministic via `RANDOM_STATE`, but training on Apple's Metal GPU is not bit-exact across runs — expect ±0.01 AUC noise.
- **Resuming tuning.** Per-subject tuning checkpoints (`weights/{subject}_tuning_results.json`) let you restart §8 mid-grid without redoing completed trials. Delete a file to force a full re-run for that subject.
- **Separate weight files.** Each subject's best model is stored in `weights/{subject}_bestmodel.weights.h5` and can be loaded independently for inference.